In [49]:
from pathlib import Path

INPUT_PARQUET   = "./processed-data/all_gps_data_no_dup.parquet"
OUTPUT_PARQUET  = Path("./processed-data/gps_rru_candidates.parquet")
CACHE_DIR       = Path("./cache")

MAX_DIST_METERS = 50

# Pre-filter bounding box (WGS84) — kotak pembungkus area Ring Road Utara
LON_MIN, LON_MAX = 110.342299, 110.433240
LAT_MIN, LAT_MAX = -7.767411, -7.742002

# Hanya bulan februari, unit epoch (s)
TS_START = 1643648400
TS_END   = 1646067599

EPSG_WGS84  = 4326
EPSG_UTM49S = 32749   # CRS metrik untuk Yogyakarta — wajib untuk distance-in-meters

SIMPANG_KRONGGAHAN = (-7.744769426425405, 110.34897889640796)
SIMPANG_JOMBOR = (-7.749221082422497, 110.36229833670288)
SIMPANG_MONJALI = (-7.751208485968954, 110.37121050305969)
SIMPANG_KENTUNGAN = (-7.754883591097026, 110.38329930140856)
SIMPANG_CONDONGCATUR = (-7.758447985710783, 110.39574179820893)
SIMPANG_UPN = (-7.761726028519824, 110.41203208408224)

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

In [50]:
from collections import Counter

import folium
import geopandas as gpd
import numpy as np
import osmnx as ox
import polars as pl
import shapely
from pyproj import Transformer
import networkx as nx
from shapely.strtree import STRtree
from folium.plugins import MeasureControl
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
TARGET_ROAD_NAMES = ("siliwangi", "padjajaran", "pajajaran")
SLEMAN_PLACE = "Sleman, Daerah Istimewa Yogyakarta, Indonesia"
GRAPH_FILENAME = "graph_osmnx_sleman.graphml"
RRU_FILENAME = "rru_edges.geojson"
CACHE_RRU = CACHE_DIR / RRU_FILENAME
 
def is_rru_name(name) -> bool:
    """OSM `name` bisa str, list[str] (multi-named segment), atau NaN."""
    if isinstance(name, list):
        return any(is_rru_name(n) for n in name)
    if not isinstance(name, str):
        return False
    lowered = name.lower()
    return any(target in lowered for target in TARGET_ROAD_NAMES)

def _remove_disconnected_segments(
    G: nx.MultiDiGraph, edge_ids: list
) -> gpd.GeoDataFrame:
    """Sisakan hanya largest weakly-connected component dari subset edge."""
    subgraph = G.edge_subgraph(edge_ids)
    largest_cc = max(nx.weakly_connected_components(subgraph), key=len)
    _, edges = ox.graph_to_gdfs(subgraph.subgraph(largest_cc))
    return edges.copy()

def _keep_main_carriageway(
    edges: gpd.GeoDataFrame, G: nx.MultiDiGraph
) -> gpd.GeoDataFrame:
    """Tahan satu carriageway saja; buang branches / slip road residu."""
    subgraph = G.edge_subgraph(edges.index.tolist()).to_undirected()
    xs = nx.get_node_attributes(subgraph, "x")
    west = min(xs, key=xs.get)
    east = max(xs, key=xs.get)

    path_nodes = nx.shortest_path(subgraph, west, east, weight="length")
    path_pairs = {frozenset(p) for p in zip(path_nodes[:-1], path_nodes[1:])}

    on_path = [frozenset([u, v]) in path_pairs for u, v, _ in edges.index]
    return edges[on_path].copy()
 
def _load_or_build_graph(cache_dir: Path) -> nx.MultiDiGraph:
    """Muat graph OSM dari cache, atau bangun baru jika belum ada."""
    cache_path = cache_dir / GRAPH_FILENAME
    if cache_path.exists():
        return ox.load_graphml(cache_path)
    graph = ox.graph_from_place(SLEMAN_PLACE, network_type="drive", simplify=False)
    ox.save_graphml(graph, cache_path)
    return graph
 
def _select_rru_trunk_edge_ids(G: nx.MultiDiGraph) -> list:
    """Pilih edge yang merupakan trunk dan mengandung nama RRU."""
    _, edges = ox.graph_to_gdfs(G)
    is_trunk = edges["highway"].apply(lambda h: "trunk" in str(h))
    is_rru = edges["name"].apply(is_rru_name)
    return edges[is_trunk & is_rru].index.tolist()

def _falsing_oneway_all_edges(edges: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Falsing oneway attribute for all edges."""
    edges["oneway"] = False
    return edges

def _filter_by_bounding_box(
    edges: gpd.GeoDataFrame, 
    lon_min: float, lon_max: float, 
    lat_min: float, lat_max: float
) -> gpd.GeoDataFrame:
    """Potong segmen jalan menggunakan spatial indexer GeoPandas (.cx)"""
    # .cx[xmin:xmax, ymin:ymax] akan mempertahankan edge yang bersinggungan 
    # atau berada di dalam area koordinat tersebut.
    filtered_edges = edges.cx[lon_min:lon_max, lat_min:lat_max]
    return filtered_edges.copy()

def load_rru_edges(cache_dir: Path, cache_rru: Path) -> gpd.GeoDataFrame:
    G = _load_or_build_graph(cache_dir)
    edge_ids = _select_rru_trunk_edge_ids(G)
    rru_edges = _remove_disconnected_segments(G, edge_ids)
    rru_edges = _keep_main_carriageway(rru_edges, G)
    rru_edges = _filter_by_bounding_box(rru_edges, LON_MIN, LON_MAX, LAT_MIN, LAT_MAX)
    rru_edges = _falsing_oneway_all_edges(rru_edges)
    rru_edges.to_file(cache_rru, driver="GeoJSON")
    return rru_edges

rru_edges = load_rru_edges(CACHE_DIR, CACHE_RRU)
print(f"RRU trunk edges: {len(rru_edges)} segmen")
 
name_counts = Counter(str(n) for n in rru_edges["name"].dropna())
for name, count in name_counts.most_common():
    print(f"  - {name}: {count}")

RRU trunk edges: 189 segmen
  - Jalan Padjajaran: 145
  - Jalan Siliwangi: 44


In [52]:
def filter_single_ping(df: pl.DataFrame) -> pl.DataFrame:
    """
    Menghapus data dengan maid yang hanya memiliki 1 ping (single-ping) 
    dan mencetak ringkasan perubahannya.
    """
    print(f"Titik : {df.height:,}")

    before_maids = df["maid"].n_unique()
    before_pings = df.height

    # Filter data: pertahankan maid yang muncul >= 2 kali
    df_filtered = df.filter(pl.len().over("maid") >= 2)

    after_maids = df_filtered["maid"].n_unique()
    after_pings = df_filtered.height

    print(f"Drop single-ping maid : {before_maids - after_maids:,} maid "
          f"({before_pings - after_pings:,} ping)")
    print(f"Sisa                  : {after_maids:,} maid, {after_pings:,} ping")

    return df_filtered

In [53]:
output_bbox_path = Path("./processed-data/output_bbox.parquet")

if output_bbox_path.exists():
    df_bb = pl.read_parquet(output_bbox_path)
    print(f"Load existing output: {df_bb.height:,} ping, {df_bb['maid'].n_unique():,} maid")
else:
    bbox_filter = (
        pl.col("latitude").is_between(LAT_MIN, LAT_MAX)
        & pl.col("longitude").is_between(LON_MIN, LON_MAX)
    )

    scan = pl.scan_parquet(INPUT_PARQUET).filter(bbox_filter)
    if TS_START is not None and TS_END is not None:
        scan = scan.filter(pl.col("timestamp").is_between(TS_START, TS_END))

    df_bb = scan.select("maid", "latitude", "longitude", "timestamp").collect()
    df_bb = filter_single_ping(df_bb)

    df_bb.write_parquet(output_bbox_path, compression="zstd")
    print(f"Saved: {output_bbox_path}")

Load existing output: 2,606,049 ping, 84,080 maid


In [54]:
def build_edge_spatial_index(
    edges_gdf,
    target_epsg: int,
) -> tuple[STRtree, np.ndarray, np.ndarray, int]:
    """Reproject `edges_gdf` ke `target_epsg` (meter-based) dan bangun R-tree.
    
    Mengembalikan tuple: (tree, geometries, edge_keys, crs_epsg)
    """
    edges_proj = edges_gdf.to_crs(epsg=target_epsg).reset_index()
    geoms = edges_proj.geometry.to_numpy()
    edge_keys = (
        edges_proj["u"].astype(str) + "_" + edges_proj["v"].astype(str)
    ).to_numpy()

    shapely.prepare(geoms)  # mutate in-place untuk percepat predicate
    tree = STRtree(geoms)

    return tree, geoms, edge_keys, target_epsg


def find_edge_candidates(
    points_df: pl.DataFrame,
    edge_index: tuple[STRtree, np.ndarray, np.ndarray, int],
    *,
    source_epsg: int,
    max_distance_m: float,
    lon_col: str = "longitude",
    lat_col: str = "latitude",
    sort_keys: list[str] | None = None,
) -> pl.DataFrame:
    """Tambahkan kolom kandidat edge ke setiap titik di `points_df`.
    
    `edge_index` adalah tuple dari build_edge_spatial_index:
    (tree, geometries, edge_keys, crs_epsg)
    """
    tree, geometries, edge_keys, crs_epsg = edge_index

    # Project titik ke CRS edges supaya distance dalam meter
    transformer = Transformer.from_crs(
        source_epsg, crs_epsg, always_xy=True
    )
    x, y = transformer.transform(
        points_df[lon_col].to_numpy(),
        points_df[lat_col].to_numpy(),
    )
    pts = shapely.points(x, y)

    # Spatial filter -> hitung jarak presisi
    pt_idx, edge_idx = tree.query(
        pts, predicate="dwithin", distance=max_distance_m
    )
    dists = shapely.distance(pts[pt_idx], geometries[edge_idx])

    per_point = (
        pl.DataFrame({
            "row_idx": pt_idx,
            "edge_key": edge_keys[edge_idx],
            "dist": np.round(dists, 2),
        })
        .sort(["row_idx", "dist"])
        .group_by("row_idx", maintain_order=True)
        .agg(
            candidate_edge_keys=pl.col("edge_key"),
            candidate_dists=pl.col("dist"),
        )
    )

    # Join kembali ke DataFrame asli
    result = (
        points_df.with_row_index("row_idx")
        .with_columns(pl.col("row_idx").cast(pl.Int64))
        .join(per_point, on="row_idx", how="inner")
        .drop("row_idx")
    )
    return result.sort(sort_keys) if sort_keys else result

In [55]:
edge_index = build_edge_spatial_index(rru_edges, target_epsg=EPSG_UTM49S)
print(f"R-tree: {len(edge_index[1])} edge")

R-tree: 189 edge


In [56]:
df_candidates = find_edge_candidates(
    df_bb,
    edge_index,
    source_epsg=EPSG_WGS84,
    max_distance_m=MAX_DIST_METERS,
    sort_keys=["maid", "timestamp"],
)

n_drop = df_bb.height - df_candidates.height
print(f"Lolos filter (>=1 edge <={MAX_DIST_METERS}m): {df_candidates.height:,}")
print(f"Drop (0 kandidat)                     : {n_drop:,}")

Lolos filter (>=1 edge <=50m): 231,570
Drop (0 kandidat)                     : 2,374,479


In [57]:
df_candidates = filter_single_ping(df_candidates)
df_candidates.write_parquet(OUTPUT_PARQUET, compression="zstd")

size_mb = OUTPUT_PARQUET.stat().st_size / 1024 / 1024
print(f"Tersimpan: {OUTPUT_PARQUET.resolve()}")
print(f"Size     : {size_mb:.1f} MB")
print(f"Schema   : {dict(df_candidates.schema)}")


Titik : 231,570
Drop single-ping maid : 1,607 maid (1,607 ping)
Sisa                  : 10,592 maid, 229,963 ping
Tersimpan: /home/repal/Github/skripsi/server/processed-data/gps_rru_candidates.parquet
Size     : 4.4 MB
Schema   : {'maid': String, 'latitude': Float64, 'longitude': Float64, 'timestamp': Int64, 'candidate_edge_keys': List(String), 'candidate_dists': List(Float64)}


In [ ]:
# ==========================================================================
# BILOCATION CHECK
# Bilocation = satu device (maid) punya >1 lokasi berbeda pada timestamp
# yang sama. Secara fisik mustahil; biasanya artifak dari:
#   (a) race condition GPS/WiFi positioning di device
#   (b) agregator merge multi-source tanpa reconcile
#   (c) clock collision / timestamp rounding
# ==========================================================================

# ---- Aggregate 1-pass: count + bbox koordinat per (maid, ts) ----
per_key = (
    df_candidates.lazy()
    .group_by(["maid", "timestamp"])
    .agg(
        n_pings=pl.len(),
        lat_min=pl.col("latitude").min(),
        lat_max=pl.col("latitude").max(),
        lon_min=pl.col("longitude").min(),
        lon_max=pl.col("longitude").max(),
    )
    .collect()
)

n_total_keys = per_key.height
biloc = per_key.filter(pl.col("n_pings") > 1)
n_biloc = biloc.height
n_rows_biloc = int(biloc["n_pings"].sum()) if n_biloc else 0

print("=" * 60)
print("BILOCATION DETECTION")
print("=" * 60)
print(f"Total unique (maid, ts) pairs     : {n_total_keys:>15,}")
print(f"Bilocation pairs (n_pings > 1)    : {n_biloc:>15,}"
      f"  ({n_biloc / n_total_keys:.4%})")
print(f"Rows involved in bilocation       : {n_rows_biloc:>15,}"
      f"  ({n_rows_biloc / df_candidates.height:.4%})")

print("\n--- Distribusi n_pings per grup bilocation ---")
print(biloc["n_pings"].describe())

# ---- Haversine (WGS84 mean radius) ----
def haversine_m(lat1, lon1, lat2, lon2):
    R_EARTH_M = 6_371_008.8
    lat1r, lat2r = np.radians(lat1), np.radians(lat2)
    dlat = lat2r - lat1r
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat / 2) ** 2
            + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2)
    return 2 * R_EARTH_M * np.arcsin(np.sqrt(a))

lat_min = biloc["lat_min"].to_numpy()
lat_max = biloc["lat_max"].to_numpy()
lon_min = biloc["lon_min"].to_numpy()
lon_max = biloc["lon_max"].to_numpy()

# Max pairwise spread via diagonal bbox (exact utk grup 2-titik)
diag_a = haversine_m(lat_min, lon_min, lat_max, lon_max)
diag_b = haversine_m(lat_min, lon_max, lat_max, lon_min)
spread_m = np.maximum(diag_a, diag_b)

biloc = biloc.with_columns(pl.Series("spread_m", spread_m))

print("=" * 60)
print("BILOCATION SPREAD (meter)")
print("=" * 60)
print(f"Jarak terjauh   : {spread_m.max():>15,.2f} m"
        f"   ({spread_m.max() / 1000:,.2f} km)")
print(f"Mean            : {spread_m.mean():>15,.2f} m")
print(f"Median          : {float(np.median(spread_m)):>15,.2f} m")
print(f"p95 / p99       : {np.percentile(spread_m, 95):>9,.2f} m / "
        f"{np.percentile(spread_m, 99):,.2f} m")

BILOCATION DETECTION
Total unique (maid, ts) pairs     :         170,915
Bilocation pairs (n_pings > 1)    :          38,816  (22.7107%)
Rows involved in bilocation       :          97,864  (42.5564%)

--- Distribusi n_pings per grup bilocation ---
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 38816.0  │
│ null_count ┆ 0.0      │
│ mean       ┆ 2.521228 │
│ std        ┆ 1.102905 │
│ min        ┆ 2.0      │
│ 25%        ┆ 2.0      │
│ 50%        ┆ 2.0      │
│ 75%        ┆ 3.0      │
│ max        ┆ 14.0     │
└────────────┴──────────┘
BILOCATION SPREAD (meter)
Jarak terjauh   :            4.95 m   (0.00 km)
Mean            :            0.55 m
Median          :            0.23 m
p95 / p99       :      2.43 m / 3.65 m


In [61]:
# Collapse semua ping pada key yang sama -> 1 titik (mean lat/lon)
group_keys = ["maid", "timestamp"]

df_collapsed_points = (
    df_candidates
    .group_by(group_keys, maintain_order=True)
    .agg(
        latitude=pl.col("latitude").mean(),
        longitude=pl.col("longitude").mean(),
        n_points_collapsed=pl.len(),   # audit: berapa row asli di grup tsb
    )
    .sort(group_keys)
)

# Recompute candidate edge untuk titik hasil collapse
# (pakai edge_index yang sudah ada; tidak perlu build ulang kalau rru_edges/CRS tidak berubah)
df_candidates_collapsed = find_edge_candidates(
    df_collapsed_points.select(["maid", "timestamp", "latitude", "longitude"]),
    edge_index,
    source_epsg=EPSG_WGS84,
    max_distance_m=MAX_DIST_METERS,
    sort_keys=group_keys,
).join(
    df_collapsed_points.select(group_keys + ["n_points_collapsed"]),
    on=group_keys,
    how="left",
)

print(f"Sebelum collapse: {df_candidates.height:,} row")
print(f"Sesudah collapse: {df_candidates_collapsed.height:,} row")
print(
    f"Grup bilocation (n_points_collapsed > 1): "
    f"{df_candidates_collapsed.filter(pl.col('n_points_collapsed') > 1).height:,}"
)

Sebelum collapse: 229,963 row
Sesudah collapse: 170,915 row
Grup bilocation (n_points_collapsed > 1): 38,816


In [62]:
df_candidates_collapsed = filter_single_ping(df_candidates_collapsed)

Titik : 170,915
Drop single-ping maid : 1,001 maid (1,001 ping)
Sisa                  : 9,591 maid, 169,914 ping
